# Final Linear Regression with Feature Engineering

This notebook focuses mainly on feature engineering.

Feature engineering means creating useful new columns from existing columns so that the model can learn better patterns.

Target column: `hours-per-week`

## 1. Import required libraries

In [ ]:
# pandas is used for data loading and manipulation
import pandas as pd

# numpy is used for numerical operations and log transformations
import numpy as np

# matplotlib is used for visualizations
import matplotlib.pyplot as plt

# train_test_split splits data into training and testing sets
from sklearn.model_selection import train_test_split

# Pipeline combines preprocessing and model training
from sklearn.pipeline import Pipeline

# ColumnTransformer applies different preprocessing for numeric and categorical columns
from sklearn.compose import ColumnTransformer

# OneHotEncoder converts categorical columns into numeric columns
# StandardScaler scales numeric values
from sklearn.preprocessing import OneHotEncoder, StandardScaler

# SimpleImputer fills missing values
from sklearn.impute import SimpleImputer

# LinearRegression is the model used for prediction
from sklearn.linear_model import LinearRegression

# Regression evaluation metrics
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

# pickle saves the final trained model
import pickle

## 2. Load dataset

In [ ]:
# Load the employee dataset
df = pd.read_csv('employee.csv')

# Display first few records
print('Dataset shape:', df.shape)
df.head()

## 3. Basic dataset understanding

In [ ]:
# Check data types and non-null counts
df.info()

In [ ]:
# Display statistical summary of numeric columns
df.describe()

## 4. Basic cleaning

In [ ]:
# Remove extra spaces from column names
df.columns = df.columns.str.strip()

# Remove extra spaces from text columns
for column in df.select_dtypes(include='object').columns:
    df[column] = df[column].str.strip()

# Replace '?' values with NaN so that missing values can be handled properly
df = df.replace('?', np.nan)

# Check missing values after replacement
print(df.isnull().sum())

## 5. Target selection

In [ ]:
# We select hours-per-week as the target column because it is numeric and continuous
# Linear Regression is suitable for predicting numeric continuous values
target = 'hours-per-week'

# Display target distribution
plt.figure(figsize=(8, 5))
plt.hist(df[target].dropna(), bins=20)
plt.title('Target Distribution: Hours Per Week')
plt.xlabel('Hours Per Week')
plt.ylabel('Employee Count')
plt.show()

## 6. Feature engineering - age group

In [ ]:
# Create age groups from age
# This converts raw age into meaningful business-friendly groups
df['age_group'] = pd.cut(
    df['age'],
    bins=[0, 25, 35, 45, 55, 65, 100],
    labels=['very_young', 'young', 'mid_age', 'senior', 'very_senior', 'elder']
)

# Check age group distribution
df['age_group'].value_counts()

## 7. Feature engineering - education level

In [ ]:
# Convert education-num into education level groups
# This gives a more interpretable version of education-num
df['education_level'] = pd.cut(
    df['education-num'],
    bins=[0, 8, 12, 16, 20],
    labels=['low', 'medium', 'high', 'very_high']
)

# Check new education level feature
df[['education-num', 'education_level']].head()

## 8. Feature engineering - marital status indicators

In [ ]:
# Create a binary column to indicate whether the person is married
# 1 means married, 0 means not married
df['is_married'] = df['marital-status'].apply(
    lambda value: 1 if isinstance(value, str) and 'Married' in value else 0
)

# Check result
df[['marital-status', 'is_married']].head()

## 9. Feature engineering - workclass indicator

In [ ]:
# Create a binary column to identify private-sector employees
# This can help the model learn workclass-related patterns
df['is_private_workclass'] = df['workclass'].apply(
    lambda value: 1 if value == 'Private' else 0
)

# Check result
df[['workclass', 'is_private_workclass']].head()

## 10. Feature engineering - native country indicator

In [ ]:
# Create a binary feature to identify employees from United States
# This simplifies high-cardinality native-country information
df['is_us_native'] = df['native-country'].apply(
    lambda value: 1 if value == 'United-States' else 0
)

# Check result
df[['native-country', 'is_us_native']].head()

## 11. Feature engineering - capital gain/loss features

In [ ]:
# Create indicator columns showing whether employee has capital gain/loss
# These convert zero/non-zero behavior into useful model signals
df['has_capital_gain'] = (df['capital-gain'] > 0).astype(int)
df['has_capital_loss'] = (df['capital-loss'] > 0).astype(int)

# Create net capital feature
df['capital_net'] = df['capital-gain'] - df['capital-loss']

# Log transformation reduces the effect of very large values
# log1p(x) means log(1 + x), useful when values can be zero
df['log_capital_gain'] = np.log1p(df['capital-gain'])
df['log_capital_loss'] = np.log1p(df['capital-loss'])

# Check engineered capital features
df[['capital-gain', 'capital-loss', 'has_capital_gain', 'has_capital_loss', 'capital_net', 'log_capital_gain', 'log_capital_loss']].head()

## 12. Feature engineering - fnlwgt transformation

In [ ]:
# fnlwgt has large values, so log transformation can make it less skewed
df['log_fnlwgt'] = np.log1p(df['fnlwgt'])

# Compare original and transformed values
df[['fnlwgt', 'log_fnlwgt']].head()

## 13. Feature engineering - interaction feature

In [ ]:
# Interaction feature combines age and education-num
# This helps the model capture combined effect of age and education
df['age_x_education'] = df['age'] * df['education-num']

# Check interaction feature
df[['age', 'education-num', 'age_x_education']].head()

## 14. Feature engineering - missing value indicator features

In [ ]:
# Missing values themselves can sometimes carry meaning
# These columns indicate whether original values were missing
for column in ['workclass', 'occupation', 'native-country']:
    df[column + '_missing'] = df[column].isnull().astype(int)

# Check missing indicator columns
df[['workclass_missing', 'occupation_missing', 'native-country_missing']].head()

## 15. Select final features

In [ ]:
# Select original useful columns and engineered columns
# We avoid using target column as an input feature
selected_features = [
    'age',
    'workclass',
    'education',
    'education-num',
    'marital-status',
    'occupation',
    'relationship',
    'race',
    'gender',
    'capital-gain',
    'capital-loss',
    'native-country',
    'income',
    'age_group',
    'education_level',
    'is_married',
    'is_private_workclass',
    'is_us_native',
    'has_capital_gain',
    'has_capital_loss',
    'capital_net',
    'log_capital_gain',
    'log_capital_loss',
    'log_fnlwgt',
    'age_x_education',
    'workclass_missing',
    'occupation_missing',
    'native-country_missing'
]

# X contains input features
X = df[selected_features]

# y contains target values
y = df[target]

print('Final feature count:', len(selected_features))
X.head()

## 16. Identify numeric and categorical features

In [ ]:
# Numeric columns need imputation and scaling
numeric_features = X.select_dtypes(include=['int64', 'float64']).columns.tolist()

# Categorical columns need imputation and one-hot encoding
categorical_features = X.select_dtypes(include=['object', 'category']).columns.tolist()

print('Numeric features:', numeric_features)
print('Categorical features:', categorical_features)

## 17. Train-test split

In [ ]:
# Split data into training and testing sets
# Training data is used to train the model
# Testing data is used to evaluate final performance
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

print('Training data:', X_train.shape)
print('Testing data:', X_test.shape)

## 18. Build preprocessing pipeline

In [ ]:
# Numeric pipeline:
# 1. Fill missing numeric values using median
# 2. Scale numeric values
numeric_pipeline = Pipeline(
    steps=[
        ('imputer', SimpleImputer(strategy='median')),
        ('scaler', StandardScaler())
    ]
)

# Categorical pipeline:
# 1. Fill missing categories using most frequent value
# 2. Convert categories into numeric columns using one-hot encoding
categorical_pipeline = Pipeline(
    steps=[
        ('imputer', SimpleImputer(strategy='most_frequent')),
        ('encoder', OneHotEncoder(handle_unknown='ignore'))
    ]
)

# Combine both pipelines
preprocessor = ColumnTransformer(
    transformers=[
        ('num', numeric_pipeline, numeric_features),
        ('cat', categorical_pipeline, categorical_features)
    ]
)

## 19. Build final Linear Regression model

In [ ]:
# Final model pipeline:
# Step 1: Preprocess data
# Step 2: Train Linear Regression model
model = Pipeline(
    steps=[
        ('preprocessor', preprocessor),
        ('regressor', LinearRegression())
    ]
)

## 20. Train the model

In [ ]:
# Train model using feature-engineered training data
model.fit(X_train, y_train)

print('Feature engineering Linear Regression model trained successfully')

## 21. Predict on test data

In [ ]:
# Predict hours-per-week for test data
y_pred = model.predict(X_test)

# Show first few predictions
print(y_pred[:10])

## 22. Evaluate model performance

In [ ]:
# Calculate evaluation metrics
mae = mean_absolute_error(y_test, y_pred)
mse = mean_squared_error(y_test, y_pred)
rmse = mse ** 0.5
r2 = r2_score(y_test, y_pred)

# Print results
print('Mean Absolute Error:', mae)
print('Mean Squared Error:', mse)
print('Root Mean Squared Error:', rmse)
print('R2 Score:', r2)

## 23. Actual vs predicted plot

In [ ]:
# This plot compares actual working hours and predicted working hours
plt.figure(figsize=(7, 5))
plt.scatter(y_test, y_pred, alpha=0.4)
plt.xlabel('Actual Hours Per Week')
plt.ylabel('Predicted Hours Per Week')
plt.title('Actual vs Predicted Hours Per Week')
plt.show()

## 24. Residual plot

In [ ]:
# Residual = actual value - predicted value
# A good regression model should have residuals randomly spread around zero
residuals = y_test - y_pred

plt.figure(figsize=(7, 5))
plt.scatter(y_pred, residuals, alpha=0.4)
plt.axhline(y=0, linestyle='--')
plt.xlabel('Predicted Hours Per Week')
plt.ylabel('Residuals')
plt.title('Residual Plot')
plt.show()

## 25. Save final model

In [ ]:
# Save the complete pipeline including preprocessing and trained model
with open('final_linear_regression_feature_engineering_model.pkl', 'wb') as file:
    pickle.dump(model, file)

print('Final feature engineering model saved successfully')